In [1]:
from snowflake.snowpark import Session

In [3]:
from snowflake.snowpark import Session

connection_parameters = {
    "account": "",
    "user": "",
    "password": "",
    "role": "",
    "warehouse": "",
    "database": "",
    "schema": ""
}

session = Session.builder.configs(connection_parameters).create()

In [4]:
session.sql("""
SELECT 
    CURRENT_ROLE(), 
    CURRENT_DATABASE(), 
    CURRENT_SCHEMA(), 
    CURRENT_WAREHOUSE()
""").show()

----------------------------------------------------------------------------------------
|"CURRENT_ROLE()"  |"CURRENT_DATABASE()"  |"CURRENT_SCHEMA()"  |"CURRENT_WAREHOUSE()"  |
----------------------------------------------------------------------------------------
|ACCOUNTADMIN      |SNOWPARK              |SAMPLE_DATA         |COMPUTE_WH             |
----------------------------------------------------------------------------------------



In [5]:
session.sql("CREATE DATABASE IF NOT EXISTS SNOWPARK").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS SNOWPARK.SAMPLE_DATA").collect()

session.sql("USE DATABASE SNOWPARK").collect()
session.sql("USE SCHEMA SAMPLE_DATA").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

[Row(status='Statement executed successfully.')]

In [6]:
session.sql("""
SELECT 
    CURRENT_ROLE(), 
    CURRENT_DATABASE(), 
    CURRENT_SCHEMA(), 
    CURRENT_WAREHOUSE()
""").show()

----------------------------------------------------------------------------------------
|"CURRENT_ROLE()"  |"CURRENT_DATABASE()"  |"CURRENT_SCHEMA()"  |"CURRENT_WAREHOUSE()"  |
----------------------------------------------------------------------------------------
|ACCOUNTADMIN      |SNOWPARK              |SAMPLE_DATA         |COMPUTE_WH             |
----------------------------------------------------------------------------------------



In [7]:
session.sql("""
CREATE OR REPLACE TABLE CAMPAIGN_SPEND (
    DATE DATE,
    CHANNEL STRING,
    TOTAL_COST FLOAT,
    CLICKS INTEGER,
    ADS_SERVED INTEGER
)
""").collect()

[Row(status='Table CAMPAIGN_SPEND successfully created.')]

In [8]:
session.sql("""
INSERT INTO CAMPAIGN_SPEND VALUES
('2024-01-01', 'search_engine', 1000, 120, 10000),
('2024-01-01', 'email', 500, 80, 7000),
('2024-01-01', 'video', 700, 60, 9000),
('2024-01-01', 'social_media', 300, 40, 5000),

('2024-02-01', 'search_engine', 1200, 150, 12000),
('2024-02-01', 'email', 600, 90, 8000),
('2024-02-01', 'video', 750, 70, 9500),
('2024-02-01', 'social_media', 350, 45, 5500),

('2024-03-01', 'search_engine', 1400, 170, 13000),
('2024-03-01', 'email', 700, 100, 8500),
('2024-03-01', 'video', 900, 85, 10000),
('2024-03-01', 'social_media', 450, 60, 6500),

('2024-04-01', 'search_engine', 1600, 190, 15000),
('2024-04-01', 'email', 800, 120, 9000),
('2024-04-01', 'video', 950, 90, 11000),
('2024-04-01', 'social_media', 500, 70, 7000),

('2024-05-01', 'search_engine', 1800, 210, 17000),
('2024-05-01', 'email', 900, 130, 9500),
('2024-05-01', 'video', 1100, 100, 12500),
('2024-05-01', 'social_media', 650, 85, 8500),

('2024-06-01', 'search_engine', 2000, 240, 20000),
('2024-06-01', 'email', 1000, 150, 11000),
('2024-06-01', 'video', 1300, 120, 14000),
('2024-06-01', 'social_media', 800, 100, 10000)
""").collect()

[Row(number of rows inserted=24)]

In [9]:
snow_df_spend = session.table("CAMPAIGN_SPEND")
snow_df_spend.show()

-----------------------------------------------------------------------
|"DATE"      |"CHANNEL"      |"TOTAL_COST"  |"CLICKS"  |"ADS_SERVED"  |
-----------------------------------------------------------------------
|2024-01-01  |search_engine  |1000.0        |120       |10000         |
|2024-01-01  |email          |500.0         |80        |7000          |
|2024-01-01  |video          |700.0         |60        |9000          |
|2024-01-01  |social_media   |300.0         |40        |5000          |
|2024-02-01  |search_engine  |1200.0        |150       |12000         |
|2024-02-01  |email          |600.0         |90        |8000          |
|2024-02-01  |video          |750.0         |70        |9500          |
|2024-02-01  |social_media   |350.0         |45        |5500          |
|2024-03-01  |search_engine  |1400.0        |170       |13000         |
|2024-03-01  |email          |700.0         |100       |8500          |
----------------------------------------------------------------

In [10]:
session.sql("""
CREATE OR REPLACE TABLE MONTHLY_REVENUE (
    YEAR INTEGER,
    MONTH INTEGER,
    REVENUE FLOAT
)
""").collect()

[Row(status='Table MONTHLY_REVENUE successfully created.')]

In [11]:
session.sql("""
INSERT INTO MONTHLY_REVENUE VALUES
(2024, 1, 10000),
(2024, 2, 12500),
(2024, 3, 14500),
(2024, 4, 16800),
(2024, 5, 19500),
(2024, 6, 23000)
""").collect()

[Row(number of rows inserted=6)]

In [12]:
session.table("MONTHLY_REVENUE").show()

--------------------------------
|"YEAR"  |"MONTH"  |"REVENUE"  |
--------------------------------
|2024    |1        |10000.0    |
|2024    |2        |12500.0    |
|2024    |3        |14500.0    |
|2024    |4        |16800.0    |
|2024    |5        |19500.0    |
|2024    |6        |23000.0    |
--------------------------------



In [13]:
from snowflake.snowpark.functions import col, sum as sum_, year, month

In [14]:
snow_df_spend = session.table("CAMPAIGN_SPEND")

spend_by_channel = (
    snow_df_spend
    .group_by(
        year(col("DATE")).alias("YEAR"),
        month(col("DATE")).alias("MONTH"),
        col("CHANNEL")
    )
    .agg(sum_(col("TOTAL_COST")).alias("TOTAL_COST"))
    .sort("YEAR", "MONTH", "CHANNEL")
)

spend_by_channel.show()

---------------------------------------------------
|"YEAR"  |"MONTH"  |"CHANNEL"      |"TOTAL_COST"  |
---------------------------------------------------
|2024    |1        |email          |500.0         |
|2024    |1        |search_engine  |1000.0        |
|2024    |1        |social_media   |300.0         |
|2024    |1        |video          |700.0         |
|2024    |2        |email          |600.0         |
|2024    |2        |search_engine  |1200.0        |
|2024    |2        |social_media   |350.0         |
|2024    |2        |video          |750.0         |
|2024    |3        |email          |700.0         |
|2024    |3        |search_engine  |1400.0        |
---------------------------------------------------



In [15]:
pivoted_spend = (
    spend_by_channel
    .pivot("CHANNEL", ["search_engine", "email", "video", "social_media"])
    .sum("TOTAL_COST")
)

pivoted_spend.show()

-----------------------------------------------------------------------------------
|"YEAR"  |"MONTH"  |"'search_engine'"  |"'email'"  |"'video'"  |"'social_media'"  |
-----------------------------------------------------------------------------------
|2024    |1        |1000.0             |500.0      |700.0      |300.0             |
|2024    |2        |1200.0             |600.0      |750.0      |350.0             |
|2024    |3        |1400.0             |700.0      |900.0      |450.0             |
|2024    |4        |1600.0             |800.0      |950.0      |500.0             |
|2024    |5        |1800.0             |900.0      |1100.0     |650.0             |
|2024    |6        |2000.0             |1000.0     |1300.0     |800.0             |
-----------------------------------------------------------------------------------



In [16]:
pivoted_spend_clean = (
    pivoted_spend
    .rename("'search_engine'", "SEARCH_ENGINE")
    .rename("'email'", "EMAIL")
    .rename("'video'", "VIDEO")
    .rename("'social_media'", "SOCIAL_MEDIA")
)

pivoted_spend_clean.show()

---------------------------------------------------------------------------
|"YEAR"  |"MONTH"  |"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |
---------------------------------------------------------------------------
|2024    |1        |1000.0           |500.0    |700.0    |300.0           |
|2024    |2        |1200.0           |600.0    |750.0    |350.0           |
|2024    |4        |1600.0           |800.0    |950.0    |500.0           |
|2024    |6        |2000.0           |1000.0   |1300.0   |800.0           |
|2024    |5        |1800.0           |900.0    |1100.0   |650.0           |
|2024    |3        |1400.0           |700.0    |900.0    |450.0           |
---------------------------------------------------------------------------



In [17]:
snow_df_revenue = session.table("MONTHLY_REVENUE")

training_df = pivoted_spend_clean.join(
    snow_df_revenue,
    on=["YEAR", "MONTH"]
)

training_df.show()

---------------------------------------------------------------------------------------
|"YEAR"  |"MONTH"  |"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |"REVENUE"  |
---------------------------------------------------------------------------------------
|2024    |1        |1000.0           |500.0    |700.0    |300.0           |10000.0    |
|2024    |2        |1200.0           |600.0    |750.0    |350.0           |12500.0    |
|2024    |3        |1400.0           |700.0    |900.0    |450.0           |14500.0    |
|2024    |4        |1600.0           |800.0    |950.0    |500.0           |16800.0    |
|2024    |5        |1800.0           |900.0    |1100.0   |650.0           |19500.0    |
|2024    |6        |2000.0           |1000.0   |1300.0   |800.0           |23000.0    |
---------------------------------------------------------------------------------------



In [18]:
features_df = (
    training_df
    .dropna()
    .drop("YEAR", "MONTH")
)

features_df.show()

--------------------------------------------------------------------
|"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |"REVENUE"  |
--------------------------------------------------------------------
|1000.0           |500.0    |700.0    |300.0           |10000.0    |
|1200.0           |600.0    |750.0    |350.0           |12500.0    |
|1400.0           |700.0    |900.0    |450.0           |14500.0    |
|1600.0           |800.0    |950.0    |500.0           |16800.0    |
|1800.0           |900.0    |1100.0   |650.0           |19500.0    |
|2000.0           |1000.0   |1300.0   |800.0           |23000.0    |
--------------------------------------------------------------------



In [19]:
features_df.write.mode("overwrite").save_as_table(
    "MARKETING_BUDGETS_FEATURES"
)

In [20]:
session.table("MARKETING_BUDGETS_FEATURES").show()

--------------------------------------------------------------------
|"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |"REVENUE"  |
--------------------------------------------------------------------
|1000.0           |500.0    |700.0    |300.0           |10000.0    |
|1200.0           |600.0    |750.0    |350.0           |12500.0    |
|1400.0           |700.0    |900.0    |450.0           |14500.0    |
|1600.0           |800.0    |950.0    |500.0           |16800.0    |
|1800.0           |900.0    |1100.0   |650.0           |19500.0    |
|2000.0           |1000.0   |1300.0   |800.0           |23000.0    |
--------------------------------------------------------------------



In [21]:
df = session.table(
    "SNOWPARK.SAMPLE_DATA.MARKETING_BUDGETS_FEATURES"
).to_pandas()

df

,SEARCH_ENGINE,EMAIL,VIDEO,SOCIAL_MEDIA,REVENUE
0,1000.0,500.0,700.0,300.0,10000.0
1,1200.0,600.0,750.0,350.0,12500.0
2,1400.0,700.0,900.0,450.0,14500.0
3,1600.0,800.0,950.0,500.0,16800.0
4,1800.0,900.0,1100.0,650.0,19500.0
5,2000.0,1000.0,1300.0,800.0,23000.0


In [22]:
X = df.drop("REVENUE", axis=1)
y = df["REVENUE"]

In [23]:
X.head()

,SEARCH_ENGINE,EMAIL,VIDEO,SOCIAL_MEDIA
0,1000.0,500.0,700.0,300.0
1,1200.0,600.0,750.0,350.0
2,1400.0,700.0,900.0,450.0
3,1600.0,800.0,950.0,500.0
4,1800.0,900.0,1100.0,650.0


In [24]:
y.head()

0    10000.0
1    12500.0
2    14500.0
3    16800.0
4    19500.0
Name: REVENUE, dtype: float64

In [25]:
from sklearn.model_selection import train_test_split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [27]:
from sklearn.linear_model import LinearRegression

In [28]:
model = LinearRegression()

In [30]:
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [31]:
y_pred = model.predict(X_test)

In [32]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [33]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE :", rmse)
print("R2 :", r2)

RMSE : 1204.1594578792333
R2 : 0.07199999999999429


In [34]:
import joblib

In [35]:
joblib.dump(model, "revenue_model.joblib")

['revenue_model.joblib']

In [36]:
model = joblib.load("revenue_model.joblib")

In [37]:
new_campaigns = [
    [2200, 1100, 1400, 900],
    [2500, 1300, 1600, 1000],
    [1800, 900, 1200, 700]
]

columns = [
    "SEARCH_ENGINE",
    "EMAIL",
    "VIDEO",
    "SOCIAL_MEDIA"
]

In [38]:
import pandas as pd

new_campaigns_df = pd.DataFrame(
    new_campaigns,
    columns=columns
)

new_campaigns_df

,SEARCH_ENGINE,EMAIL,VIDEO,SOCIAL_MEDIA
0,2200,1100,1400,900
1,2500,1300,1600,1000
2,1800,900,1200,700


In [39]:
predictions = model.predict(new_campaigns_df)

In [40]:
new_campaigns_df["PREDICTED_REVENUE"] = predictions

In [41]:
new_campaigns_df

,SEARCH_ENGINE,EMAIL,VIDEO,SOCIAL_MEDIA,PREDICTED_REVENUE
0,2200,1100,1400,900,25500.0
1,2500,1300,1600,1000,30860.0
2,1800,900,1200,700,20500.0


In [42]:
predictions_snow_df = session.create_dataframe(new_campaigns_df)

In [43]:
predictions_snow_df.show()


------------------------------------------------------------------------------
|"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |"PREDICTED_REVENUE"  |
------------------------------------------------------------------------------
|2200             |1100     |1400     |900             |25500.000000000004   |
|2500             |1300     |1600     |1000            |30860.000000000004   |
|1800             |900      |1200     |700             |20499.999999999996   |
------------------------------------------------------------------------------



In [44]:
predictions_snow_df.write.mode("overwrite").save_as_table(
    "CAMPAIGN_REVENUE_PREDICTIONS"
)

In [45]:
session.table("CAMPAIGN_REVENUE_PREDICTIONS").show()

------------------------------------------------------------------------------
|"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |"PREDICTED_REVENUE"  |
------------------------------------------------------------------------------
|2200             |1100     |1400     |900             |25500.000000000004   |
|2500             |1300     |1600     |1000            |30860.000000000004   |
|1800             |900      |1200     |700             |20499.999999999996   |
------------------------------------------------------------------------------



In [46]:
session.sql("""
SELECT *
FROM CAMPAIGN_REVENUE_PREDICTIONS
ORDER BY PREDICTED_REVENUE DESC
""").show()

------------------------------------------------------------------------------
|"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |"PREDICTED_REVENUE"  |
------------------------------------------------------------------------------
|2500             |1300     |1600     |1000            |30860.000000000004   |
|2200             |1100     |1400     |900             |25500.000000000004   |
|1800             |900      |1200     |700             |20499.999999999996   |
------------------------------------------------------------------------------



In [47]:
from sklearn.ensemble import RandomForestRegressor

In [48]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [49]:
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [50]:
rf_pred = rf_model.predict(X_test)

In [51]:
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest RMSE :", rf_rmse)
print("Random Forest R2 :", rf_r2)

Random Forest RMSE : 4468.400161131498
Random Forest R2 : -11.778624


In [52]:
joblib.dump(rf_model, "revenue_random_forest_model.joblib")

['revenue_random_forest_model.joblib']

In [53]:
rf_model = joblib.load("revenue_random_forest_model.joblib")

In [54]:
new_campaigns_df["RF_PREDICTED_REVENUE"] = rf_model.predict(
    new_campaigns_df[columns]
)

new_campaigns_df

,SEARCH_ENGINE,EMAIL,VIDEO,SOCIAL_MEDIA,PREDICTED_REVENUE,RF_PREDICTED_REVENUE
0,2200,1100,1400,900,25500.0,21730.0
1,2500,1300,1600,1000,30860.0,21730.0
2,1800,900,1200,700,20500.0,19543.0


In [55]:
rf_predictions_snow_df = session.create_dataframe(new_campaigns_df)

rf_predictions_snow_df.write.mode("overwrite").save_as_table(
    "CAMPAIGN_REVENUE_PREDICTIONS_RF"
)

In [56]:
session.table("CAMPAIGN_REVENUE_PREDICTIONS_RF").show()

-------------------------------------------------------------------------------------------------------
|"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |"PREDICTED_REVENUE"  |"RF_PREDICTED_REVENUE"  |
-------------------------------------------------------------------------------------------------------
|2200             |1100     |1400     |900             |25500.000000000004   |21730.0                 |
|2500             |1300     |1600     |1000            |30860.000000000004   |21730.0                 |
|1800             |900      |1200     |700             |20499.999999999996   |19543.0                 |
-------------------------------------------------------------------------------------------------------



In [57]:
session.sql("""
CREATE OR REPLACE TABLE ROI_PRED (
    SEARCH_ENGINE FLOAT,
    EMAIL FLOAT,
    VIDEO FLOAT,
    SOCIAL_MEDIA FLOAT,
    PREDICTED_REVENUE FLOAT
)
""").collect()

[Row(status='Table ROI_PRED successfully created.')]

In [58]:
session.sql("""
INSERT INTO ROI_PRED (
    SEARCH_ENGINE,
    EMAIL,
    VIDEO,
    SOCIAL_MEDIA,
    PREDICTED_REVENUE
)
SELECT 
    SEARCH_ENGINE,
    EMAIL,
    VIDEO,
    SOCIAL_MEDIA,
    PREDICTED_REVENUE
FROM CAMPAIGN_REVENUE_PREDICTIONS
""").collect()

[Row(number of rows inserted=3)]

In [59]:
session.table("ROI_PRED").show()

------------------------------------------------------------------------------
|"SEARCH_ENGINE"  |"EMAIL"  |"VIDEO"  |"SOCIAL_MEDIA"  |"PREDICTED_REVENUE"  |
------------------------------------------------------------------------------
|2200.0           |1100.0   |1400.0   |900.0           |25500.000000000004   |
|2500.0           |1300.0   |1600.0   |1000.0          |30860.000000000004   |
|1800.0           |900.0    |1200.0   |700.0           |20499.999999999996   |
------------------------------------------------------------------------------



In [60]:
session.sql("SHOW TABLES").show()

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"created_on"                      |"name"                           |"database_name"  |"schema_name"  |"kind"     |"comment"  |"cluster_by"  |"rows"  |"bytes"  |"owner"       |"retention_time"  |"automatic_clustering"  |"change_tracking"  |"search_optimization"  |"search_optimization_progress"  |"search_optimization_bytes"  |"is_external"  |"enable_schema_evolution"  |"owner_role_type"  |"is_event"  |"is_hybrid"  |"is_iceberg"  |"is_dynamic"  |"is_immutable"  |"is_interactive"  |"row_timest

In [61]:
session.sql("SHOW STAGES").show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"created_on"                      |"name"                          |"database_name"  |"schema_name"  |"url"  |"has_credentials"  |"has_encryption_key"  |"owner"       |"comment"  |"region"  |"type"              |"cloud"  |"notification_channel"  |"storage_integration"  |"endpoint"  |"owner_role_type"  |"directory_enabled"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-05-10 13:1

In [62]:
session.sql("""
SELECT 
    CURRENT_ROLE(), 
    CURRENT_DATABASE(), 
    CURRENT_SCHEMA(), 
    CURRENT_WAREHOUSE()
""").show()

----------------------------------------------------------------------------------------
|"CURRENT_ROLE()"  |"CURRENT_DATABASE()"  |"CURRENT_SCHEMA()"  |"CURRENT_WAREHOUSE()"  |
----------------------------------------------------------------------------------------
|ACCOUNTADMIN      |SNOWPARK              |SAMPLE_DATA         |COMPUTE_WH             |
----------------------------------------------------------------------------------------



In [63]:
session.sql("USE DATABASE SNOWPARK").collect()
session.sql("USE SCHEMA SAMPLE_DATA").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

[Row(status='Statement executed successfully.')]

In [64]:
session.sql("DROP TABLE IF EXISTS CAMPAIGN_REVENUE_PREDICTIONS").collect()
session.sql("DROP TABLE IF EXISTS CAMPAIGN_REVENUE_PREDICTIONS_RF").collect()
session.sql("DROP TABLE IF EXISTS ROI_PRED").collect()
session.sql("DROP TABLE IF EXISTS MARKETING_BUDGETS_FEATURES").collect()
session.sql("DROP TABLE IF EXISTS CAMPAIGN_SPEND").collect()
session.sql("DROP TABLE IF EXISTS MONTHLY_REVENUE").collect()

[Row(status='MONTHLY_REVENUE successfully dropped.')]